# Chapter 14: Reinforcement Learning
**Part IV — Specialist Domains**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

MDPs, Q-learning, DQN, policy gradients, PPO, and real-world applications.

## Learning Objectives
See Chapter 14 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Q-learning on a simple GridWorld


In [ ]:
# Q-learning on a simple GridWorld
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 4×4 grid: agent starts at (0,0), goal at (3,3), hole at (1,1) and (2,2)
GRID  = 4
GOAL  = (3, 3)
HOLES = {(1, 1), (2, 2)}
ACTIONS = [(0,1),(0,-1),(1,0),(-1,0)]  # right, left, down, up
ACT_NAMES = ['R','L','D','U']

def step(s, a):
    r, c = s
    dr, dc = ACTIONS[a]
    nr, nc = r + dr, c + dc
    if not (0 <= nr < GRID and 0 <= nc < GRID):  # wall
        return s, -0.1, False
    ns = (nr, nc)
    if ns == GOAL:    return ns, +10.0, True
    if ns in HOLES:   return ns, -10.0, True
    return ns, -0.1, False

# Q-learning
np.random.seed(42)
Q = np.zeros((GRID, GRID, 4))
alpha, gamma, eps = 0.1, 0.95, 0.3
rewards_per_ep = []

for ep in range(3000):
    s = (0, 0); total_r = 0
    for _ in range(50):
        if np.random.rand() < eps:
            a = np.random.randint(4)
        else:
            a = Q[s[0], s[1]].argmax()
        ns, r, done = step(s, a)
        Q[s[0],s[1],a] += alpha * (r + gamma * Q[ns[0],ns[1]].max() - Q[s[0],s[1],a])
        s = ns; total_r += r
        if done: break
    rewards_per_ep.append(total_r)
    eps = max(0.05, eps * 0.999)

# Plot learning curve and learned policy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Smoothed rewards
window = 100
smoothed = np.convolve(rewards_per_ep, np.ones(window)/window, 'valid')
ax1.plot(smoothed, color='#2E75B6', lw=2)
ax1.set_xlabel('Episode'); ax1.set_ylabel('Smoothed total reward')
ax1.set_title(f'Q-learning: reward curve (100-episode moving avg)', fontsize=11)
ax1.axhline(0, color='k', lw=0.5, linestyle='--'); ax1.grid(alpha=0.3)

# Learned policy grid
best_acts = Q.argmax(axis=2)
arrows = {0:'→',1:'←',2:'↓',3:'↑'}
for r in range(GRID):
    for c in range(GRID):
        color = '#EAFAF1' if (r,c)==GOAL else '#FDEDEC' if (r,c) in HOLES else '#EBF5FB'
        ax2.add_patch(patches.Rectangle((c,GRID-1-r), 1, 1, facecolor=color, edgecolor='#888'))
        label = 'GOAL' if (r,c)==GOAL else 'HOLE' if (r,c) in HOLES else arrows[best_acts[r,c]]
        ax2.text(c+0.5, GRID-1-r+0.5, label, ha='center', va='center', fontsize=14,
                 fontweight='bold', color='#1F3864')
ax2.set_xlim(0,GRID); ax2.set_ylim(0,GRID)
ax2.set_xticks([]); ax2.set_yticks([])
ax2.set_title('Learned greedy policy (Q.argmax)', fontsize=11)
ax2.text(0.5, -0.15, 'Agent starts at top-left ↑', transform=ax2.transAxes, ha='center', fontsize=9)

plt.suptitle('Q-Learning on a 4×4 GridWorld', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch14_qlearning.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 📚 Review Questions

See Chapter 14 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 14 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*